# MySQL `company` Database — Create & Populate

Creates the five normalized operational tables (customers, products,
transactions, events, campaigns) and loads them from the source CSVs.

**events** includes simulated UTM tracking columns (`utm_campaign`,
`utm_source`, `utm_medium`) added to enable attribution between web
behavior and ad-platform campaigns (see `add_utm_to_events.py`).

## 1. Imports

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

## 2. Connection

In [ ]:
schema = 
host = 
user = 
password = 
port = 

con = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
engine = create_engine(con)
print("engine ready")

## 3. Table Definitions (DDL)

Five normalized tables. `events` includes the three UTM columns.

In [ ]:
DDL = {

"customers": """
CREATE TABLE IF NOT EXISTS customers (
    customer_id          INT          NOT NULL,
    signup_date          DATE         NOT NULL,
    country              CHAR(2)      NOT NULL,
    age                  TINYINT UNSIGNED,
    gender                VARCHAR(20),
    loyalty_tier         VARCHAR(20),
    acquisition_channel  VARCHAR(50),
    PRIMARY KEY (customer_id),
    INDEX idx_cust_country (country)
) ENGINE=InnoDB
""",

"products": """
CREATE TABLE IF NOT EXISTS products (
    product_id   INT           NOT NULL,
    category     VARCHAR(50)   NOT NULL,
    brand        VARCHAR(50),
    base_price   DECIMAL(10,2) NOT NULL,
    launch_date  DATE,
    is_premium   TINYINT(1)    NOT NULL DEFAULT 0,
    PRIMARY KEY (product_id),
    INDEX idx_prod_category (category)
) ENGINE=InnoDB
""",

"transactions": """
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id    BIGINT        NOT NULL,
    order_ts          DATETIME      NOT NULL,
    customer_id       INT           NOT NULL,
    product_id        INT           NOT NULL,
    quantity          INT           NOT NULL,
    discount_applied  DECIMAL(10,2) NOT NULL DEFAULT 0.00,
    gross_revenue     DECIMAL(12,2) NOT NULL,
    campaign_id       INT           NOT NULL DEFAULT 0,
    refund_flag       TINYINT(1)    NOT NULL DEFAULT 0,
    PRIMARY KEY (transaction_id),
    INDEX idx_tx_customer (customer_id),
    INDEX idx_tx_product  (product_id),
    INDEX idx_tx_ts       (order_ts),
    CONSTRAINT fk_tx_customer FOREIGN KEY (customer_id) REFERENCES customers (customer_id),
    CONSTRAINT fk_tx_product  FOREIGN KEY (product_id)  REFERENCES products (product_id)
) ENGINE=InnoDB
""",

"events": """
CREATE TABLE IF NOT EXISTS events (
    event_id              BIGINT       NOT NULL,
    event_ts              DATETIME     NOT NULL,
    customer_id           INT,
    session_id            BIGINT       NOT NULL,
    event_type            VARCHAR(30)  NOT NULL,
    product_id            INT,
    device_type           VARCHAR(20),
    traffic_source        VARCHAR(50),
    campaign_id            INT          NOT NULL DEFAULT 0,
    page_category          VARCHAR(20),
    session_duration_sec  DECIMAL(10,2),
    experiment_group      VARCHAR(20),
    utm_campaign           VARCHAR(50),
    utm_source             VARCHAR(30),
    utm_medium              VARCHAR(30),
    PRIMARY KEY (event_id),
    INDEX idx_ev_session   (session_id),
    INDEX idx_ev_customer  (customer_id),
    INDEX idx_ev_product   (product_id),
    INDEX idx_ev_type      (event_type),
    INDEX idx_ev_ts        (event_ts),
    INDEX idx_ev_utm_campaign (utm_campaign),
    CONSTRAINT fk_ev_customer FOREIGN KEY (customer_id) REFERENCES customers (customer_id),
    CONSTRAINT fk_ev_product  FOREIGN KEY (product_id)  REFERENCES products (product_id)
) ENGINE=InnoDB
""",

"campaigns": """
CREATE TABLE IF NOT EXISTS campaigns (
    campaign_row_id  BIGINT         NOT NULL AUTO_INCREMENT,
    spend_date       DATE           NOT NULL,
    platform         VARCHAR(30)    NOT NULL,
    campaign_id      VARCHAR(20)    NOT NULL,
    adset_id         VARCHAR(20),
    ad_id            VARCHAR(20),
    campaign_name    VARCHAR(255),
    adset_name       VARCHAR(255),
    ad_name          VARCHAR(255),
    spend            DECIMAL(12,2)  NOT NULL DEFAULT 0.00,
    impressions      BIGINT         NOT NULL DEFAULT 0,
    clicks           INT            NOT NULL DEFAULT 0,
    conversions      INT            NOT NULL DEFAULT 0,
    country          VARCHAR(2),
    objective        VARCHAR(20),
    campaign_clean   VARCHAR(50),
    audience         VARCHAR(30),
    creative_type    VARCHAR(20),
    year             SMALLINT,
    month            TINYINT,
    PRIMARY KEY (campaign_row_id),
    INDEX idx_camp_date     (spend_date),
    INDEX idx_camp_platform (platform),
    INDEX idx_camp_campaign (campaign_id),
    INDEX idx_camp_clean    (campaign_clean)
) ENGINE=InnoDB
""",

}

## 4. Create the Tables

In [ ]:
with engine.connect() as conn:
    for name, ddl in DDL.items():
        conn.execute(text(ddl))
        print(f"created: {name}")
    conn.commit()

print("\nall tables created")

## 5. Load Data

Order matters: parent tables (customers, products) load before child
tables (transactions, events) that reference them via foreign key.

### 5.1 Customers & Products (parents)

In [ ]:
DATA = "~/Desktop/mrkt_pipeline/datasets"

# ---- customers ----
df = pd.read_csv(f"{DATA}/customers.csv")
df["signup_date"] = pd.to_datetime(df["signup_date"]).dt.date
df.to_sql("customers", engine, if_exists="append", index=False, chunksize=5000)
print(f"customers: {len(df):,}")

# ---- products ----
df = pd.read_csv(f"{DATA}/products.csv")
df["launch_date"] = pd.to_datetime(df["launch_date"]).dt.date
df.to_sql("products", engine, if_exists="append", index=False, chunksize=5000)
print(f"products: {len(df):,}")

### 5.2 Transactions (child)

In [ ]:
df = pd.read_csv(f"{DATA}/transactions.csv")
df["product_id"] = df["product_id"].astype(float).astype("Int64")   # 1630.0 -> 1630
df = df.rename(columns={"timestamp": "order_ts"})
df["order_ts"] = pd.to_datetime(df["order_ts"])

before = len(df)
df = df.dropna(subset=["product_id", "customer_id"])
print(f"dropped {before - len(df)} rows with null product_id/customer_id")

df.to_sql("transactions", engine, if_exists="append", index=False, chunksize=5000)
print(f"transactions: {len(df):,}")

### 5.3 Events (child, 2M rows — includes UTM columns)

Loads from `events_with_utm.csv` (the output of `add_utm_to_events.py`),
which adds `utm_campaign` / `utm_source` / `utm_medium` — simulated
UTM tracking that links web behavior to ad-platform campaigns.

In [ ]:
df = pd.read_csv(f"{DATA}/events_with_utm.csv")
df["product_id"] = df["product_id"].astype(float).astype("Int64")
df = df.rename(columns={"timestamp": "event_ts"})
df["event_ts"] = pd.to_datetime(df["event_ts"])

df.to_sql("events", engine, if_exists="append", index=False, chunksize=10000)
print(f"events: {len(df):,}")
print(f"  with utm_campaign: {df['utm_campaign'].notna().sum():,}")

### 5.4 Campaigns (cleaned ad-spend)

In [ ]:
df = pd.read_csv(f"{DATA}/ad_spend_cleaned.csv")
df = df.rename(columns={"date": "spend_date"})
df["spend_date"] = pd.to_datetime(df["spend_date"]).dt.date

df.to_sql("campaigns", engine, if_exists="append", index=False, chunksize=5000)
print(f"campaigns: {len(df):,}")

## 6. Verify Row Counts

In [ ]:
for t in ["customers", "products", "transactions", "events", "campaigns"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", engine)["n"][0]
    print(f"{t:<15}: {n:,}")